# 05 — Causal ML Feature Attribution + Cold-Start LTV

**Phase 4 — Week 4**

**Goals:**
1. Identify which features **CAUSE** high LTV (not just correlate)
2. Estimate heterogeneous treatment effects (CATE) per customer segment
3. Build firmographic cold-start LTV prior for zero-transaction customers
4. Save causal lever recommendations for the dashboard

**Method:** EconML LinearDML (Double Machine Learning)
- Step 1: Residualise Y (LTV) and T (treatment) on controls X using ML
- Step 2: Regress Y-residual on T-residual → unbiased causal effect
- Cross-fitting (k=5) removes overfitting bias

**Treatments tested:**
- `onboarding_completed` (proxy: days_to_second_purchase ≤ 7)
- `high_value_first_purchase` (proxy: first_purchase_amount in top 40%)
- `multi_category_buyer` (proxy: unique_categories ≥ 3)
- `fast_repeat_buyer` (proxy: days_to_second_purchase ≤ 30)
- `high_frequency` (proxy: frequency ≥ 5)
- `international_buyer` (proxy: multi_country = True)


In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import polars as pl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from rich import print as rprint
import plotly.io as pio
pio.templates.default = 'plotly_white'

from backend.config import settings
from backend.data.load_data import load_uci_csv
from backend.db.supabase_client import SupabaseClient
from backend.features.rfm import (
    clean_transactions, assign_product_categories,
    assign_amount_buckets, make_calibration_holdout_split, RFMPipeline
)
from backend.ml.causal_model import (
    CausalLTVPipeline, prepare_causal_dataset,
    TREATMENT_DEFINITIONS, CONTROL_FEATURES
)
from backend.ml.cold_start import (
    ColdStartScorer, build_firmographic_lookup
)
from backend.ml.causal_dag import get_dag_records, DAG_NODE_METADATA
from backend.ml.causal_heterogeneous import (
    compute_heterogeneity_report, find_high_leverage_customers
)

print('✓ Imports OK')

✓ Imports OK


In [2]:
# ── Config ────────────────────────────────────────────────────
CAUSAL_MODEL_VERSION = 'causal_v1'
USE_CAUSAL_FOREST    = False   # True = slower but richer CI
CV_FOLDS             = 5
SAVE_TO_DB           = True
USE_WANDB            = True
OBSERVATION_MONTHS   = 6
HOLDOUT_MONTHS       = 6

print(f'Model version: {CAUSAL_MODEL_VERSION}')
print(f'Use CausalForest: {USE_CAUSAL_FOREST}')

Model version: causal_v1
Use CausalForest: False


## 1. Load Data & Prepare Causal Dataset

In [3]:
# Load and clean
raw_df  = load_uci_csv(settings.UCI_CSV_PATH)
cleaned = clean_transactions(raw_df)
cleaned = assign_product_categories(cleaned)
cleaned = assign_amount_buckets(cleaned)

# Split
calibration, holdout, obs_end, holdout_end = make_calibration_holdout_split(
    cleaned,
    observation_months=int(OBSERVATION_MONTHS),
    holdout_months=int(HOLDOUT_MONTHS),
    date_col='invoice_date',
)

# RFM with labels
rfm_pipe = RFMPipeline(calibration, observation_end_date=obs_end)
rfm_df   = rfm_pipe.compute()
rfm_df   = rfm_pipe.compute_ltv_labels(holdout, rfm_df, horizon_months=12)

print(f'RFM customers: {len(rfm_df):,}')
print(f'Non-zero LTV 12m: {(rfm_df["actual_ltv_12m"] > 0).sum():,}')


2026-04-29 14:14:51.372 | INFO     | backend.data.load_data:load_uci_csv:75 - Loading UCI dataset from: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\backend\data\raw\OnlineRetail.csv
2026-04-29 14:14:51.373 | INFO     | backend.data.load_data:load_uci_csv:93 - Reading CSV with DuckDB for robust date parsing...
2026-04-29 14:14:56.682 | INFO     | backend.data.load_data:load_uci_csv:130 - Loaded 541909 rows — 8 columns — date range 2010-12-01 08:26:00+00:00 to 2011-12-09 12:50:00+00:00
2026-04-29 14:14:56.682 | INFO     | backend.features.rfm:clean_transactions:54 - Cleaning transactions — 541909 raw rows
2026-04-29 14:14:56.709 | INFO     | backend.features.rfm:clean_transactions:76 - After cleaning: 397884 rows (73.4% kept)
2026-04-29 14:14:56.740 | INFO     | backend.features.rfm:make_calibration_holdout_split:157 - Split → calibration 144541 rows (≤2011-05-30), holdout 225975 rows (2011-05-30 – 2011-11-26)
2026-04-29 14:14:56.741 | INFO     | backend.feat

RFM customers: 2,708
Non-zero LTV 12m: 1,875


In [4]:
# Prepare causal dataset
df_causal, treatments, controls = prepare_causal_dataset(rfm_df)

print(f'Causal dataset: {len(df_causal):,} customers')
print(f'Treatments ({len(treatments)}): {treatments}')
print(f'Controls ({len(controls)}): {controls}')
print(f'\nTreatment prevalences:')
for t in treatments:
    col = f't_{t}'
    pct = 100 * df_causal[col].mean()
    rprint(f'  {t:30s}: {pct:.1f}%')

2026-04-29 14:14:56.842 | INFO     | backend.ml.causal_model:prepare_causal_dataset:202 - Causal dataset: 2708 customers, 6 treatments, 9 controls


Causal dataset: 2,708 customers
Treatments (6): ['onboarding_completed', 'high_value_first_purchase', 'multi_category_buyer', 'fast_repeat_buyer', 'high_frequency', 'international_buyer']
Controls (9): ['frequency', 'recency_days', 't_days', 'monetary_avg', 'monetary_std', 'orders_count', 'avg_days_between_orders', 'unique_products', 'unique_categories']

Treatment prevalences:


onboarding_completed          : 6.2%

high_value_first_purchase     : 40.0%

multi_category_buyer          : 18.8%

fast_repeat_buyer             : 19.0%

high_frequency                : 8.2%

international_buyer           : 0.1%

## 2. Causal DAG Visualisation

In [5]:
# Show DAG node types
dag_data = pd.DataFrame([
    {'node': k, 'type': v['type'], 'description': v['description']}
    for k, v in DAG_NODE_METADATA.items()
])

node_colors = {'treatment': '#e74c3c', 'outcome': '#27ae60', 'confounder': '#3498db'}
fig = px.scatter(
    dag_data, x='type', y='node',
    color='type', color_discrete_map=node_colors,
    hover_data=['description'],
    title='Causal DAG — Node Types',
    height=450,
)
fig.show()

display(dag_data.groupby('type').size().reset_index(name='count'))

,type,count
0,confounder,7
1,outcome,1
2,treatment,5


## 3. Exploratory Correlation Analysis (Pre-Causal Baseline)

In [6]:
# Naive (non-causal) correlation of each treatment with LTV
# This is what we'd get WITHOUT causal adjustment — potentially confounded
correlations = []
for t in treatments:
    col = f't_{t}'
    treated   = df_causal[df_causal[col] == 1]['log_ltv']
    untreated = df_causal[df_causal[col] == 0]['log_ltv']
    naive_diff = float(np.expm1(treated.mean()) - np.expm1(untreated.mean()))
    correlations.append({
        'treatment':       t,
        'naive_diff_ltv':  naive_diff,
        'n_treated':       len(treated),
        'n_untreated':     len(untreated),
        'mean_ltv_treated':  float(np.expm1(treated.mean())),
        'mean_ltv_control':  float(np.expm1(untreated.mean())),
    })

corr_df = pd.DataFrame(correlations).sort_values('naive_diff_ltv', ascending=False)
display(corr_df)

fig = px.bar(
    corr_df, x='treatment', y='naive_diff_ltv',
    title='Naive (Unadjusted) LTV Difference by Treatment — BEFORE Causal Correction',
    color='naive_diff_ltv', color_continuous_scale='RdYlGn',
    labels={'naive_diff_ltv': 'LTV Difference (£)'},
)
fig.show()

,treatment,naive_diff_ltv,n_treated,n_untreated,mean_ltv_treated,mean_ltv_control
4,high_frequency,2575.704658,222,2486,2650.609149,74.904491
5,international_buyer,1398.017941,4,2704,1498.188001,100.170060
2,multi_category_buyer,1022.356644,509,2199,1080.111301,57.754657
0,onboarding_completed,748.404830,168,2540,835.755061,87.350231
3,fast_repeat_buyer,569.542048,515,2193,634.574858,65.032810
1,high_value_first_purchase,157.963367,1083,1625,217.859983,59.896616


## 4. Fit Double ML Causal Models

In [7]:
rprint(f'[bold blue]Fitting {len(treatments)} Double ML models...[/bold blue]')
rprint(f'This may take 5–15 minutes depending on dataset size.')

pipeline = CausalLTVPipeline(
    model_version=CAUSAL_MODEL_VERSION,
    use_causal_forest=USE_CAUSAL_FOREST,
    outcome_col='actual_ltv_12m',
    cv_folds=CV_FOLDS,
)

pipeline.fit(rfm_df)
rprint(f'\n✓ Causal pipeline fitted ({len(pipeline.estimators)} treatments)')

Fitting 6 Double ML models...

This may take 5–15 minutes depending on dataset size.

2026-04-29 14:15:01.533 | INFO     | backend.ml.causal_model:prepare_causal_dataset:202 - Causal dataset: 2708 customers, 6 treatments, 9 controls
2026-04-29 14:15:01.534 | INFO     | backend.ml.causal_model:fit:517 - Fitting 6 causal models (n_customers=2708, n_controls=9)
2026-04-29 14:15:01.536 | INFO     | backend.ml.causal_model:fit:282 - Fitting LinearDML for treatment='onboarding_completed' (n=2708, treatment_mean=0.062)
2026-04-29 14:15:03.754 | INFO     | backend.ml.causal_model:fit:310 -   ATE(onboarding_completed) = 0.25  stderr = 1.11
2026-04-29 14:15:03.759 | INFO     | backend.ml.causal_model:fit:282 - Fitting LinearDML for treatment='high_value_first_purchase' (n=2708, treatment_mean=0.400)
2026-04-29 14:15:05.958 | INFO     | backend.ml.causal_model:fit:310 -   ATE(high_value_first_purchase) = 0.51  stderr = 0.36
2026-04-29 14:15:05.964 | INFO     | backend.ml.causal_model:fit:282 - Fitting LinearDML for treatment='multi_category_buyer' (n=2708, treatment_mean=0.188)
20

✓ Causal pipeline fitted (6 treatments)

## 5. Average Treatment Effects

In [8]:
# Get ATE summary
effects_df = pipeline.get_treatment_effects_summary()
rprint('\n[bold]Average Treatment Effects (causal):[/bold]')
display(effects_df.select(['treatment_name', 'ate', 'ate_lower_ci', 'ate_upper_ci',
                             'ate_pvalue', 'is_significant', 'effect_description']))

# Compare naive vs causal
comparison = pd.merge(
    corr_df[['treatment', 'naive_diff_ltv']],
    effects_df.to_pandas()[['treatment_name', 'ate']].rename(columns={'treatment_name': 'treatment', 'ate': 'causal_ate'}),
    on='treatment', how='inner'
)
comparison['confounding_bias'] = comparison['naive_diff_ltv'] - comparison['causal_ate']
rprint('\n[bold]Confounding Bias (naive - causal):[/bold]')
display(comparison)

Average Treatment Effects (causal):

treatment_name,ate,ate_lower_ci,ate_upper_ci,ate_pvalue,is_significant,effect_description
str,f64,f64,f64,f64,bool,str
"""multi_category_buyer""",1.092104,-9.963743,12.147951,0.846481,false,"""Customer purchased from 3+ pro…"
"""fast_repeat_buyer""",0.548189,-1.186865,2.283244,0.535745,false,"""Made second purchase within 30…"
"""high_value_first_purchase""",0.507195,-0.190267,1.204658,0.154067,false,"""First purchase was in top 40% …"
"""onboarding_completed""",0.250147,-1.927889,2.428182,0.821897,false,"""Customer completed onboarding …"
"""high_frequency""",-0.017902,-65.750148,65.714345,0.999574,false,"""Made 5+ purchases in observati…"
"""international_buyer""",-0.998282,-18.187843,16.191278,0.909375,false,"""Purchased from non-UK country"""


Confounding Bias (naive - causal):

,treatment,naive_diff_ltv,causal_ate,confounding_bias
0,high_frequency,2575.704658,-0.017902,2575.722559
1,international_buyer,1398.017941,-0.998282,1399.016223
2,multi_category_buyer,1022.356644,1.092104,1021.264540
3,onboarding_completed,748.404830,0.250147,748.154684
4,fast_repeat_buyer,569.542048,0.548189,568.993859
5,high_value_first_purchase,157.963367,0.507195,157.456172


In [9]:
# ATE waterfall chart
ate_sorted = effects_df.sort('ate', descending=True)

fig = go.Figure()
colors = [
    '#27ae60' if ate > 0 else '#e74c3c'
    for ate in ate_sorted['ate'].to_list()
]
fig.add_trace(go.Bar(
    y=ate_sorted['treatment_name'].to_list(),
    x=ate_sorted['ate'].to_list(),
    orientation='h',
    marker_color=colors,
    error_x=dict(
        type='data',
        array=[(u - l) / 2 for u, l in zip(
            ate_sorted['ate_upper_ci'].to_list(),
            ate_sorted['ate_lower_ci'].to_list()
        )],
        visible=True
    ),
))
fig.add_vline(x=0, line_color='black', line_width=1)
fig.update_layout(
    title='Causal Average Treatment Effects on LTV (£)',
    xaxis_title='ATE (£ increase in 12m LTV)',
    height=400,
)
fig.show()

## 6. CATE Distributions (Heterogeneous Effects)

In [10]:
# CATE distributions per treatment
n_treatments = len(pipeline.cate_results)
cols = min(3, n_treatments)
rows_n = (n_treatments + cols - 1) // cols

treatment_list = list(pipeline.cate_results.keys())
fig = make_subplots(
    rows=rows_n, cols=cols,
    subplot_titles=treatment_list,
)
for i, t_name in enumerate(treatment_list):
    row_i = i // cols + 1
    col_i = i %  cols + 1
    cate  = pipeline.cate_results[t_name]
    fig.add_trace(go.Histogram(x=cate.tolist(), nbinsx=50, showlegend=False,
                                marker_color='steelblue', opacity=0.7),
                  row=row_i, col=col_i)
    fig.add_vline(x=0, line_dash='dash', line_color='red', row=row_i, col=col_i)  # pyright: ignore[reportArgumentType]

fig.update_layout(height=200 * rows_n + 80, title='CATE Distributions per Treatment')
fig.show()

In [11]:
# Heterogeneity report
het_report = compute_heterogeneity_report(pipeline.cate_results, rfm_df)
rprint('\n[bold]Heterogeneity Report:[/bold]')
display(het_report)

Heterogeneity Report:

treatment_name,ate,ate_std,ate_median,ate_p25,ate_p75,pct_positive_cate,mean_cate_top25,heterogeneity_index,n_customers
str,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""high_frequency""",8.8940e7,1.8391e8,0.009354,-0.999975,683378.416006,50.073855,3.5572e8,2.067782,2708
"""multi_category_buyer""",4.4418e6,4.4841e7,0.009561,-0.912245,11.908087,50.073855,1.7767e7,10.095275,2708
"""international_buyer""",3.8398e6,4.0021e7,-0.999235,-0.999999,2.964788,29.135894,1.5359e7,10.422719,2708
"""onboarding_completed""",194440.408122,9.3526e6,0.06635,-0.287034,0.805527,54.025111,777761.869145,48.100003,2708
"""fast_repeat_buyer""",193.643681,9944.839409,0.90736,0.217146,1.07291,79.837518,773.202607,51.356385,2708
"""high_value_first_purchase""",0.971623,3.832958,0.448507,-0.008195,1.387027,74.483013,3.090502,3.944903,2708


In [12]:
# Scatter: CATE vs monetary_avg (do rich customers benefit more?)
best_treatment = str(effects_df['treatment_name'][0])  # highest ATE
cate_vals = pipeline.cate_results[best_treatment]

pipeline_df = pipeline._df
assert pipeline_df is not None, 'pipeline._df is None; run pipeline.fit(...) first'

scatter_df = pd.DataFrame({
    'cate': cate_vals,
    'monetary_avg': pipeline_df['monetary_avg'].values,
    'frequency': pipeline_df['frequency'].values,
    'actual_ltv_12m': pipeline_df['actual_ltv_12m'].values,
})
p99_cate = np.percentile(np.abs(cate_vals), 99)
scatter_df = scatter_df[np.abs(scatter_df['cate']) <= p99_cate]

fig = px.scatter(
    scatter_df.sample(min(2000, len(scatter_df))),
    x='monetary_avg', y='cate',
    color='frequency',
    color_continuous_scale='Viridis',
    opacity=0.5,
    title=f'CATE({best_treatment}) vs Average Order Value',
    labels={'monetary_avg': 'Avg Order Value (GBP)', 'cate': 'CATE (GBP)'},
    trendline='ols',
)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.show()


## 7. High-Leverage Customers

In [13]:
# Customers with highest total causal uplift potential
pipeline_df = pipeline._df
assert pipeline_df is not None, 'pipeline._df is None; run pipeline.fit(...) first'

customer_ids = pipeline_df['customer_id'].tolist()

high_leverage = find_high_leverage_customers(
    pipeline.cate_results,
    customer_ids,
    min_total_uplift=200,
    top_n=50,
)
rprint('[bold]Top 10 High-Leverage Customers:[/bold]')
display(high_leverage.head(10))

fig = px.histogram(
    high_leverage.to_pandas(), x='total_uplift', nbins=30,
    title='Total Causal Uplift Distribution (Top 50 customers)',
    labels={'total_uplift': 'Total Uplift (GBP)'},
)
fig.show()


Top 10 High-Leverage Customers:

customer_id,total_uplift,cate_onboarding_completed,cate_high_value_first_purchase,cate_multi_category_buyer,cate_fast_repeat_buyer,cate_high_frequency,cate_international_buyer
str,f64,f64,f64,f64,f64,f64,f64
"""12346""",1.4560e9,4.8517e8,-1.0,4.8517e8,517607.104645,-1.0,4.8517e8
"""12971""",1.4555e9,3.601295,-0.740288,4.8517e8,30.94611,4.8517e8,4.8517e8
"""16422""",1.4555e9,2.248963,-0.643948,4.8517e8,5.572947,4.8517e8,4.8517e8
"""13798""",1.1848e9,3.051476,-0.648523,4.8517e8,13.52414,4.8517e8,2.1450e8
"""15749""",1.0102e9,3.9821e7,-0.96411,4.8517e8,6.605618,-1.0,4.8517e8
"""16029""",9.7039e8,-0.662015,2.842668,63886.19391,1.400695,4.8517e8,4.8517e8
"""14911""",9.7033e8,88.024665,-0.840501,4.8517e8,1841.344799,4.8517e8,-1.0
"""15061""",9.7033e8,-0.808062,7.017618,1396.872275,0.126562,4.8517e8,4.8517e8
"""14606""",9.7033e8,129.237694,-0.973739,4.8517e8,1130.778388,4.8517e8,-1.0


## 8. Lever Recommendations per Customer

In [14]:
# Top lever per customer
lever_df = pipeline.get_top_lever_per_customer()
rprint(f'\n[bold]Lever distribution (which lever is most impactful):[/bold]')

lever_counts = (
    lever_df.group_by('top_lever')
    .agg(pl.len().alias('n_customers'),
          pl.col('top_lever_effect_usd').mean().alias('avg_effect'))
    .sort('n_customers', descending=True)
)
display(lever_counts)

fig = px.bar(
    lever_counts.to_pandas(),
    x='top_lever', y='n_customers',
    color='avg_effect', color_continuous_scale='Greens',
    title='Primary Causal Lever by Customer Count',
    labels={'n_customers': 'Customers', 'top_lever': 'Lever'},
)
fig.show()

Lever distribution (which lever is most impactful):

top_lever,n_customers,avg_effect
str,u32,f64
"""high_frequency""",1207,1.9506e8
"""fast_repeat_buyer""",574,0.962316
"""multi_category_buyer""",494,2.2305e7
"""international_buyer""",313,1.1852e7
"""high_value_first_purchase""",99,0.163739
"""onboarding_completed""",21,2.3103e7


## 9. Cold-Start LTV Firmographic Lookup Table

In [15]:
# Build firmographic lookup table from CATE estimates
rprint('[bold blue]Building firmographic cold-start LTV table...[/bold blue]')

pipeline_df = pipeline._df
assert pipeline_df is not None, 'pipeline._df is None; run pipeline.fit(...) first'

lookup_df = build_firmographic_lookup(
    rfm_df=rfm_df,
    cate_per_customer=pipeline.cate_results,
    customer_ids=pipeline_df['customer_id'].tolist(),
    causal_model_version=CAUSAL_MODEL_VERSION,
)

print(f'Firmographic lookup table: {len(lookup_df):,} slices')
display(lookup_df.sort('ltv_36m_estimate', descending=True).head(10))


Building firmographic cold-start LTV table...

2026-04-29 14:15:22.851 | INFO     | backend.ml.cold_start:build_firmographic_lookup:181 - Firmographic lookup table: 66 slices built from 2708 customers


Firmographic lookup table: 66 slices


vertical,company_size,channel,plan_tier,ltv_36m_estimate,ci_lower,ci_upper,n_customers,cate_effect,computed_at,causal_model_version
str,str,str,str,f64,f64,f64,i32,f64,str,str
"""other""","""smb""","""organic""","""free""",1.5106e8,30678.151748,2.4258e8,7,3.0212e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""
"""other""","""smb""","""direct""","""free""",1.3519e8,72.0,2.4258e8,9,2.7038e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""
"""other""","""enterprise""","""referral""","""enterprise_trial""",1.3479e8,816.554385,4.8520e8,9,2.6956e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""
"""other""","""smb""","""paid_search""","""professional""",1.3281e8,2.6564e7,2.3905e8,2,2.6561e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""
"""other""","""enterprise""","""direct""","""enterprise_trial""",1.1967e8,746.996,4.3503e8,14,2.3932e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""
"""other""","""mid_market""","""organic""","""free""",1.1595e8,0.306748,2.4258e8,59,2.3190e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""
"""other""","""mid_market""","""paid_search""","""enterprise_trial""",1.1048e8,77.726359,3.6699e8,7,2.2096e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""
"""other""","""mid_market""","""paid_social""","""free""",1.0744e8,0.080008,2.4258e8,98,2.1488e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""
"""other""","""mid_market""","""direct""","""enterprise_trial""",1.0397e8,733.563097,3.3962e8,7,2.0793e8,"""2026-04-29T08:45:22.850965+00:…","""causal_v1"""


In [16]:
# LTV heatmap by vertical × company_size
heatmap_data = (
    lookup_df
    .group_by(['vertical', 'company_size'])
    .agg(pl.col('ltv_36m_estimate').mean().alias('avg_ltv'))
    .sort('avg_ltv', descending=True)
)

pivot = heatmap_data.pivot(
    values='avg_ltv', index='vertical', on='company_size', aggregate_function='first'
)
verticals = pivot['vertical'].to_list()
sizes     = [c for c in pivot.columns if c != 'vertical']
z         = pivot.select(sizes).to_numpy()

fig = go.Figure(go.Heatmap(
    z=z, x=sizes, y=verticals,
    colorscale='RdYlGn',
    text=[[f'£{v:.0f}' if v else '' for v in row] for row in z.tolist()],
    texttemplate='%{text}',
))
fig.update_layout(
    title='Cold-Start LTV Prior: Vertical × Company Size (£)',
    height=450,
)
fig.show()

In [17]:
# Test the cold-start scorer
scorer = ColdStartScorer(None)  # no DB needed if we load from memory
scorer._table = lookup_df

test_cases = [
    {'vertical': 'healthcare', 'company_size': 'enterprise', 'channel': 'paid_search', 'plan_tier': 'enterprise_trial'},
    {'vertical': 'ecommerce',  'company_size': 'smb',        'channel': 'organic',     'plan_tier': 'free'},
    {'vertical': 'fintech',    'company_size': 'mid_market', 'channel': 'paid_social', 'plan_tier': 'professional'},
]

rprint('\n[bold]Cold-Start Scoring Tests:[/bold]')
for tc in test_cases:
    result = scorer.score(**tc)
    rprint(f"  {tc['vertical']} / {tc['company_size']} / {tc['plan_tier']}")
    rprint(f"    LTV 36m: £{result['ltv_36m']:.0f}  CI: [£{result['ci_lower_36m']:.0f}, £{result['ci_upper_36m']:.0f}]")
    rprint(f"    Segment: {result['segment']}  Max CAC: £{result['recommended_max_cac']:.0f}")
    rprint(f"    Match:   {result['match_quality']}\n")

Cold-Start Scoring Tests:

healthcare / enterprise / enterprise_trial

LTV 36m: £42268948  CI: [£4495, £110482061]

Segment: champions  Max CAC: £16907579

Match:   global_average

...

Match:   global_average

fintech / mid_market / professional

LTV 36m: £42268948  CI: [£4495, £110482061]

Segment: champions  Max CAC: £16907579

Match:   global_average

## 10. W&B Logging

In [18]:
wandb_run_id = None

if USE_WANDB:
    try:
        import wandb

        pipeline_df = pipeline._df
        assert pipeline_df is not None, 'pipeline._df is None; run pipeline.fit(...) first'

        run = wandb.init(
            project=settings.WANDB_PROJECT,
            name='week4_causal_ml',
            tags=['week4', 'causal', 'dml', 'cold_start'],
            config={
                'model_version': CAUSAL_MODEL_VERSION,
                'estimator': 'LinearDML',
                'cv_folds': CV_FOLDS,
                'n_treatments': len(pipeline.estimators),
                'n_controls': len(pipeline._controls),
                'n_customers': len(pipeline_df),
                'outcome_col': 'actual_ltv_12m',
            }
        )
        if run is None:
            raise RuntimeError('wandb.init(...) returned None')

        wandb_run_id = run.id

        for row in effects_df.iter_rows(named=True):
            wandb.log({
                f'ate_{row["treatment_name"]}': row['ate'],
                f'pvalue_{row["treatment_name"]}': row['ate_pvalue'],
                f'sig_{row["treatment_name"]}': row['is_significant'],
            })

        wandb.log({'treatment_effects': wandb.Table(dataframe=effects_df.to_pandas())})
        wandb.log({'firmographic_lookup': wandb.Table(dataframe=lookup_df.to_pandas())})

        wandb.finish()
        rprint(f'W&B logged (run_id={wandb_run_id})')
    except Exception as e:
        rprint(f'W&B logging failed: {e}')


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rakshitkaintura2005 (rakshitkaintura2005-sir-m-visvesvaraya-institute-of-tech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


ate_fast_repeat_buyer,▁
ate_high_frequency,▁
ate_high_value_first_purchase,▁
ate_international_buyer,▁
ate_multi_category_buyer,▁
ate_onboarding_completed,▁
pvalue_fast_repeat_buyer,▁
pvalue_high_frequency,▁
pvalue_high_value_first_purchase,▁
pvalue_international_buyer,▁
+2,...


W&B logged (run_id=hncidd0c)

In [19]:
import inspect
from backend.ml.causal_model import CausalLTVPipeline
print(inspect.getsource(CausalLTVPipeline.save).splitlines()[10:25])


['        effects_df = self.get_treatment_effects_summary()', '        estimator_type = "CausalForest" if self.use_causal_forest else "DML"', '        # Normalize legacy names to match DB constraint values.', '        estimator_type = {', '            "LinearDML": "DML",', '            "CausalForestDML": "CausalForest",', '        }.get(estimator_type, estimator_type)', '        db_client.bulk_upsert("causal_model_registry", [{', '            "model_version":     self.model_version,', '            "trained_at":        datetime.now(timezone.utc).isoformat(),', '            "outcome_variable":  self.outcome_col,', '            "estimator_type":    estimator_type,', '            "n_treatments":      len(self.estimators),', '            "n_controls":        len(self._controls),', '            "n_customers":       len(self._df),']


## 11. Save to Supabase

In [20]:
if SAVE_TO_DB:
    import json
    import uuid
    from datetime import datetime, timezone

    pipeline_df = pipeline._df
    assert pipeline_df is not None, "pipeline._df is None; run pipeline.fit(...) first"

    db = SupabaseClient(use_service_role=True)
    assert db.health_check(), "DB health check failed"

    run_id = str(uuid.uuid4())[:8]

    print("Saving causal pipeline results...")
    pipeline.save(db, pipeline_run_id=run_id, wandb_run_id=wandb_run_id)

    print("Saving causal DAG...")
    dag_nodes, dag_edges = get_dag_records(CAUSAL_MODEL_VERSION)
    db.bulk_upsert(
        "causal_dag_nodes",
        dag_nodes,
        conflict_columns=["model_version", "node_name"],
    )
    db.bulk_upsert(
        "causal_dag_edges",
        dag_edges,
        conflict_columns=["model_version", "from_node", "to_node"],
    )
    print(f"Saved {len(dag_nodes)} DAG nodes, {len(dag_edges)} edges")

    print("Saving firmographic LTV table...")
    scorer_db = ColdStartScorer(db)
    n_slices = scorer_db.save_table(lookup_df, db)
    print(f"{n_slices} firmographic slices saved")

    db.bulk_upsert(
        "pipeline_runs",
        [{
            "run_id": run_id,
            "pipeline_name": "causal_ml",
            "status": "success",
            "started_at": datetime.now(timezone.utc).isoformat(),
            "records_processed": len(pipeline_df),
            "metadata": json.dumps({
                "model_version": CAUSAL_MODEL_VERSION,
                "n_treatments": len(pipeline.estimators),
                "firmographic_slices": len(lookup_df),
                "wandb_run_id": wandb_run_id,
            }),
            "wandb_run_id": wandb_run_id,
        }],
        conflict_columns=["run_id"],
    )

    print(f"All causal data saved to Supabase (run_id={run_id})")
else:
    print("Skipping DB save (SAVE_TO_DB=False)")


2026-04-29 14:16:13.496 | INFO     | backend.db.supabase_client:get_supabase_admin_client:49 - Supabase admin client initialised (service-role)
2026-04-29 14:16:13.538 | INFO     | backend.db.supabase_client:get_db_engine:79 - SQLAlchemy sync engine created — pool_size=5, max_overflow=10
2026-04-29 14:16:15.882 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into causal_model_registry (1 rows)
2026-04-29 14:16:15.918 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 1 rows into causal_model_registry


Saving causal pipeline results...


2026-04-29 14:16:16.467 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into causal_treatment_effects (6 rows)
2026-04-29 14:16:16.544 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 6 rows into causal_treatment_effects
2026-04-29 14:16:16.545 | INFO     | backend.ml.causal_model:save:699 - Saved 6 treatment effects
2026-04-29 14:16:46.108 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/33 into customer_cate (500 rows)
2026-04-29 14:17:19.508 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 2/33 into customer_cate (500 rows)
2026-04-29 14:17:53.946 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 3/33 into customer_cate (500 rows)
2026-04-29 14:18:28.301 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 4/33 into customer_cate (500 rows)
2026-04-29 14:19:07.829 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 5/33

Saving causal DAG...


2026-04-29 14:37:06.047 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into causal_dag_nodes (13 rows)
2026-04-29 14:37:06.119 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 13 rows into causal_dag_nodes
2026-04-29 14:37:07.025 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into causal_dag_edges (14 rows)
2026-04-29 14:37:07.078 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 14 rows into causal_dag_edges


Saved 13 DAG nodes, 14 edges
Saving firmographic LTV table...


2026-04-29 14:37:11.483 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into firmographic_ltv (66 rows)
2026-04-29 14:37:11.558 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 66 rows into firmographic_ltv
2026-04-29 14:37:11.560 | INFO     | backend.ml.cold_start:save_table:343 - Saved 66 firmographic LTV rows


66 firmographic slices saved


2026-04-29 14:37:11.815 | DEBUG    | backend.db.supabase_client:bulk_upsert:239 - Upserted batch 1/1 into pipeline_runs (1 rows)
2026-04-29 14:37:11.879 | INFO     | backend.db.supabase_client:bulk_upsert:247 - bulk_upsert → 1 rows into pipeline_runs


All causal data saved to Supabase (run_id=51bfa2ce)


In [21]:
pipeline_df = pipeline._df
assert pipeline_df is not None, 'pipeline._df is None; run pipeline.fit(...) first'

print('=== Week 4 Summary ===')
print(f'  Model version:         {CAUSAL_MODEL_VERSION}')
print(f'  Treatments fitted:     {len(pipeline.estimators)}')
print(f'  Customers analysed:    {len(pipeline_df):,}')
print(f'  Firmographic slices:   {len(lookup_df):,}')
print('Top 3 Causal Levers (by ATE):')
for row in effects_df.head(3).iter_rows(named=True):
    sig = '(significant)' if row['is_significant'] else '(not significant)'
    print(f"  {row['treatment_name']:30s}: GBP {row['ate']:>8.2f}  {sig}")
print(f'Cold-start scorer: {len(lookup_df)} firmographic slices')
print('Next: notebooks/06_fusion_evaluation.ipynb (Phase 5 - Week 5)')


=== Week 4 Summary ===
  Model version:         causal_v1
  Treatments fitted:     6
  Customers analysed:    2,708
  Firmographic slices:   66
Top 3 Causal Levers (by ATE):
  multi_category_buyer          : GBP     1.09  (not significant)
  fast_repeat_buyer             : GBP     0.55  (not significant)
  high_value_first_purchase     : GBP     0.51  (not significant)
Cold-start scorer: 66 firmographic slices
Next: notebooks/06_fusion_evaluation.ipynb (Phase 5 - Week 5)
